# Mask Detection - Exploratory Data Analysis

This notebook performs preliminary data analysis on the face mask detection dataset.
It explores the dataset structure, class distribution, and visualizes sample images with annotations.

## 1. Import Libraries

In [ ]:
import os
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import Counter
import seaborn as sns

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Define Dataset Paths

In [ ]:
# Define paths to dataset
IMAGES_DIR = 'face_mask_detection/images'
ANNOTATIONS_DIR = 'face_mask_detection/annotations'

# Check if directories exist
images_exist = os.path.isdir(IMAGES_DIR)
annotations_exist = os.path.isdir(ANNOTATIONS_DIR)

print(f"Images directory exists: {images_exist}")
print(f"Annotations directory exists: {annotations_exist}")

if images_exist:
    num_images = len([f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"Number of images: {num_images}")

if annotations_exist:
    num_annotations = len([f for f in os.listdir(ANNOTATIONS_DIR) if f.endswith('.xml')])
    print(f"Number of annotations: {num_annotations}")

## 3. Parse Pascal VOC Annotations

In [ ]:
def parse_xml_annotation(xml_path):
    """
    Parse Pascal VOC XML annotation file.
    Returns filename, image size, and list of objects with classes and bounding boxes.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    # Get image filename and size
    filename = root.find('filename').text
    size = root.find('size')
    width = int(size.find('width').text)
    height = int(size.find('height').text)
    
    # Parse objects
    objects = []
    for obj in root.findall('object'):
        class_name = obj.find('name').text
        bndbox = obj.find('bndbox')
        bbox = {
            'class': class_name,
            'xmin': int(bndbox.find('xmin').text),
            'ymin': int(bndbox.find('ymin').text),
            'xmax': int(bndbox.find('xmax').text),
            'ymax': int(bndbox.find('ymax').text)
        }
        objects.append(bbox)
    
    return filename, (width, height), objects

# Test parsing if annotations exist
if annotations_exist:
    sample_annotation = os.listdir(ANNOTATIONS_DIR)[0]
    sample_path = os.path.join(ANNOTATIONS_DIR, sample_annotation)
    filename, size, objects = parse_xml_annotation(sample_path)
    print(f"Sample annotation: {sample_annotation}")
    print(f"Image filename: {filename}")
    print(f"Image size: {size}")
    print(f"Number of objects: {len(objects)}")
    print(f"First object: {objects[0] if objects else 'No objects'}")

## 4. Load and Analyze Dataset

In [ ]:
# Load all annotations
dataset_info = []
class_counts = Counter()
image_sizes = []
objects_per_image = []

if annotations_exist:
    for annotation_file in os.listdir(ANNOTATIONS_DIR):
        if annotation_file.endswith('.xml'):
            xml_path = os.path.join(ANNOTATIONS_DIR, annotation_file)
            try:
                filename, size, objects = parse_xml_annotation(xml_path)
                
                # Count objects and classes
                num_objects = len(objects)
                objects_per_image.append(num_objects)
                image_sizes.append(size)
                
                for obj in objects:
                    class_counts[obj['class']] += 1
                
                dataset_info.append({
                    'annotation_file': annotation_file,
                    'image_file': filename,
                    'width': size[0],
                    'height': size[1],
                    'num_objects': num_objects
                })
            except Exception as e:
                print(f"Error parsing {annotation_file}: {e}")
    
    # Create DataFrame
    df = pd.DataFrame(dataset_info)
    print(f"\nDataset loaded successfully!")
    print(f"Total images: {len(df)}")
    print(f"\nDataset info:")
    print(df.head())
    print(f"\nDataset statistics:")
    print(df.describe())
else:
    print("Annotations directory not found. Please download the dataset first.")

## 5. Class Distribution Analysis

In [ ]:
if annotations_exist and class_counts:
    # Display class counts
    print("Class Distribution:")
    for class_name, count in sorted(class_counts.items()):
        print(f"  {class_name}: {count}")
    
    # Visualize class distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart
    classes = list(class_counts.keys())
    counts = list(class_counts.values())
    axes[0].bar(classes, counts, color=['#2ecc71', '#e74c3c', '#f39c12'])
    axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Number of Objects')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for i, (class_name, count) in enumerate(zip(classes, counts)):
        axes[0].text(i, count + 50, str(count), ha='center', va='bottom', fontweight='bold')
    
    # Pie chart
    axes[1].pie(counts, labels=classes, autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c', '#f39c12'])
    axes[1].set_title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## 6. Image Size Analysis

In [ ]:
if image_sizes:
    widths = [size[0] for size in image_sizes]
    heights = [size[1] for size in image_sizes]
    areas = [w * h for w, h in image_sizes]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Width distribution
    axes[0].hist(widths, bins=20, color='#3498db', edgecolor='black')
    axes[0].set_title('Image Width Distribution', fontweight='bold')
    axes[0].set_xlabel('Width (pixels)')
    axes[0].set_ylabel('Count')
    
    # Height distribution
    axes[1].hist(heights, bins=20, color='#9b59b6', edgecolor='black')
    axes[1].set_title('Image Height Distribution', fontweight='bold')
    axes[1].set_xlabel('Height (pixels)')
    axes[1].set_ylabel('Count')
    
    # Image area distribution
    axes[2].hist(areas, bins=20, color='#e67e22', edgecolor='black')
    axes[2].set_title('Image Area Distribution', fontweight='bold')
    axes[2].set_xlabel('Area (pixels²)')
    axes[2].set_ylabel('Count')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nImage Size Statistics:")
    print(f"Width  - Min: {min(widths)}, Max: {max(widths)}, Mean: {np.mean(widths):.0f}")
    print(f"Height - Min: {min(heights)}, Max: {max(heights)}, Mean: {np.mean(heights):.0f}")
    print(f"Area   - Min: {min(areas)}, Max: {max(areas)}, Mean: {np.mean(areas):.0f}")

## 7. Objects Per Image Analysis

In [ ]:
if objects_per_image:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(objects_per_image, bins=range(1, max(objects_per_image) + 2), 
                 color='#16a085', edgecolor='black', align='left')
    axes[0].set_title('Objects Per Image Distribution', fontweight='bold')
    axes[0].set_xlabel('Number of Objects')
    axes[0].set_ylabel('Count')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Box plot
    axes[1].boxplot(objects_per_image, vert=True)
    axes[1].set_title('Objects Per Image - Box Plot', fontweight='bold')
    axes[1].set_ylabel('Number of Objects')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nObjects Per Image Statistics:")
    print(f"Min: {min(objects_per_image)}")
    print(f"Max: {max(objects_per_image)}")
    print(f"Mean: {np.mean(objects_per_image):.2f}")
    print(f"Median: {np.median(objects_per_image):.2f}")
    print(f"Std Dev: {np.std(objects_per_image):.2f}")

## 8. Visualize Sample Images with Annotations

In [ ]:
def visualize_image_with_annotations(image_path, objects, title=""):
    """
    Visualize an image with bounding boxes and class labels.
    """
    try:
        img = Image.open(image_path)
        fig, ax = plt.subplots(1, figsize=(10, 8))
        ax.imshow(img)
        
        # Color map for classes
        color_map = {
            'with_mask': '#2ecc71',      # Green
            'without_mask': '#e74c3c',   # Red
            'mask_weared_incorrect': '#f39c12'  # Orange
        }
        
        # Draw bounding boxes
        for obj in objects:
            class_name = obj['class']
            color = color_map.get(class_name, '#95a5a6')
            
            rect = patches.Rectangle(
                (obj['xmin'], obj['ymin']),
                obj['xmax'] - obj['xmin'],
                obj['ymax'] - obj['ymin'],
                linewidth=2,
                edgecolor=color,
                facecolor='none'
            )
            ax.add_patch(rect)
            
            # Add class label
            ax.text(obj['xmin'], obj['ymin'] - 5, class_name,
                   bbox=dict(boxstyle='round', facecolor=color, alpha=0.7),
                   color='white', fontweight='bold', fontsize=10)
        
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Error visualizing image: {e}")

# Visualize random samples
if annotations_exist and len(dataset_info) > 0:
    import random
    
    # Get 3 random samples
    samples = random.sample(dataset_info, min(3, len(dataset_info)))
    
    for sample in samples:
        image_path = os.path.join(IMAGES_DIR, sample['image_file'])
        annotation_path = os.path.join(ANNOTATIONS_DIR, sample['annotation_file'])
        
        if os.path.exists(image_path):
            _, _, objects = parse_xml_annotation(annotation_path)
            title = f"{sample['image_file']} ({sample['num_objects']} objects)"
            visualize_image_with_annotations(image_path, objects, title)
        else:
            print(f"Image not found: {image_path}")

## 9. Summary and Insights

In [ ]:
if annotations_exist:
    print("=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(f"Total Images: {len(df)}")
    print(f"Total Annotated Objects: {sum(class_counts.values())}")
    print(f"\nClass Distribution:")
    for class_name in sorted(class_counts.keys()):
        count = class_counts[class_name]
        percentage = (count / sum(class_counts.values())) * 100
        print(f"  {class_name}: {count} ({percentage:.1f}%)")
    print(f"\nImage Dimensions:")
    print(f"  Width:  {min(widths)} - {max(widths)} pixels (avg: {np.mean(widths):.0f})")
    print(f"  Height: {min(heights)} - {max(heights)} pixels (avg: {np.mean(heights):.0f})")
    print(f"\nObjects per Image:")
    print(f"  Min: {min(objects_per_image)}, Max: {max(objects_per_image)}, Avg: {np.mean(objects_per_image):.2f}")
    print("=" * 60)
else:
    print("\nPlease download the dataset and place it in:")
    print(f"  Images: {IMAGES_DIR}")
    print(f"  Annotations: {ANNOTATIONS_DIR}")